# Getting Started: Noise Scan for H2

This notebook performs a simple **noise scan** for the hydrogen molecule
**H2** using the packaged **VQE** workflow.

Goals:

- run the same VQE setup across a range of noise strengths
- compare depolarizing and amplitude-damping noise
- plot final energy as a function of noise level
- inspect convergence traces at a few representative settings

This is the basic noisy-VQE workflow in the repository.

In [ ]:
from __future__ import annotations

import numpy as np
import matplotlib.pyplot as plt

from common.hamiltonian import get_exact_spectrum
from vqe.core import run_vqe

## Why a noise scan?

In noiseless simulation, VQE targets the ideal variational energy for a given
ansatz and optimizer.

In noisy simulation, gate errors and relaxation effects can shift the returned
energy and alter the optimization trajectory.

A simple way to study this is to keep the molecule and optimizer fixed and vary
the noise strength.

## Exact reference energy

For `H2`, we use the exact ground-state energy as a common reference.

In [ ]:
exact_spectrum = np.asarray(get_exact_spectrum("H2"), dtype=float)
exact_spectrum = np.sort(exact_spectrum)

exact_ground_energy = float(exact_spectrum[0])
exact_ground_energy

## Fixed VQE settings

We keep the variational setup fixed across the scan.

In [ ]:
molecule = "H2"
ansatz_name = "UCCSD"
optimizer_name = "Adam"
steps = 60
stepsize = 0.2
seed = 0

print("Molecule :", molecule)
print("Ansatz   :", ansatz_name)
print("Optimizer:", optimizer_name)

## Noise grids

We scan:

- depolarizing probability with amplitude damping fixed at zero
- amplitude-damping probability with depolarizing noise fixed at zero

This keeps the two effects easy to interpret.

In [ ]:
noise_levels = np.array([0.00, 0.01, 0.02, 0.05, 0.10], dtype=float)
noise_levels

## Depolarizing-noise scan

In [ ]:
dep_results = []
dep_final_energies = []

for p in noise_levels:
    res = run_vqe(
        molecule=molecule,
        ansatz_name=ansatz_name,
        optimizer_name=optimizer_name,
        steps=steps,
        stepsize=stepsize,
        seed=seed,
        noisy=True,
        depolarizing_prob=float(p),
        amplitude_damping_prob=0.0,
        force=True,
        plot=False,
    )

    energy = float(res["energy"])
    dep_results.append(res)
    dep_final_energies.append(energy)

    print(f"Depolarizing p = {p:.3f} -> E = {energy:.10f} Ha")

In [ ]:
dep_final_energies = np.asarray(dep_final_energies, dtype=float)
dep_final_energies

## Amplitude-damping-noise scan

In [ ]:
amp_results = []
amp_final_energies = []

for p in noise_levels:
    res = run_vqe(
        molecule=molecule,
        ansatz_name=ansatz_name,
        optimizer_name=optimizer_name,
        steps=steps,
        stepsize=stepsize,
        seed=seed,
        noisy=True,
        depolarizing_prob=0.0,
        amplitude_damping_prob=float(p),
        force=True,
        plot=False,
    )

    energy = float(res["energy"])
    amp_results.append(res)
    amp_final_energies.append(energy)

    print(f"Amplitude damping p = {p:.3f} -> E = {energy:.10f} Ha")

In [ ]:
amp_final_energies = np.asarray(amp_final_energies, dtype=float)
amp_final_energies

## Final energies vs noise strength

We compare the two noise models against the exact ground-state reference.

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(noise_levels, dep_final_energies, marker="o", label="Depolarizing")
plt.plot(noise_levels, amp_final_energies, marker="o", label="Amplitude damping")
plt.axhline(exact_ground_energy, linestyle="--", label="Exact ground energy")
plt.xlabel("Noise probability")
plt.ylabel("Final energy [Ha]")
plt.title("Noisy VQE energy for H2")
plt.grid(True)
plt.legend()
plt.show()

## Energy error vs noise strength

Absolute error makes the noise effect easier to compare directly.

In [ ]:
dep_abs_errors = np.abs(dep_final_energies - exact_ground_energy)
amp_abs_errors = np.abs(amp_final_energies - exact_ground_energy)

plt.figure(figsize=(8, 4))
plt.plot(noise_levels, dep_abs_errors, marker="o", label="Depolarizing")
plt.plot(noise_levels, amp_abs_errors, marker="o", label="Amplitude damping")
plt.xlabel("Noise probability")
plt.ylabel(r"$|E_{\mathrm{final}} - E_{\mathrm{exact}}|$ [Ha]")
plt.title("Noisy VQE absolute error for H2")
plt.grid(True)
plt.legend()
plt.show()

## Convergence traces at representative noise levels

Final energies are useful, but the optimization trajectories also matter.

We inspect:

- noiseless
- mild noise
- stronger noise

In [ ]:
representative_levels = [0.00, 0.02, 0.10]
rep_indices = [int(np.where(np.isclose(noise_levels, p))[0][0]) for p in representative_levels]

In [ ]:
plt.figure(figsize=(8, 4))
for idx in rep_indices:
    p = float(noise_levels[idx])
    trace = np.asarray(dep_results[idx]["energies"], dtype=float)
    plt.plot(np.arange(len(trace)), trace, marker="o", label=f"Depolarizing p={p:.2f}")

plt.axhline(exact_ground_energy, linestyle="--", label="Exact ground energy")
plt.xlabel("Iteration")
plt.ylabel("Energy [Ha]")
plt.title("VQE convergence under depolarizing noise for H2")
plt.grid(True)
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(8, 4))
for idx in rep_indices:
    p = float(noise_levels[idx])
    trace = np.asarray(amp_results[idx]["energies"], dtype=float)
    plt.plot(np.arange(len(trace)), trace, marker="o", label=f"Amplitude damping p={p:.2f}")

plt.axhline(exact_ground_energy, linestyle="--", label="Exact ground energy")
plt.xlabel("Iteration")
plt.ylabel("Energy [Ha]")
plt.title("VQE convergence under amplitude damping for H2")
plt.grid(True)
plt.legend()
plt.show()

## Tabulated results

In [ ]:
print(f"{'Noise p':<10} {'Depolarizing E [Ha]':>22} {'Amplitude damping E [Ha]':>28}")
print("-" * 64)
for p, e_dep, e_amp in zip(noise_levels, dep_final_energies, amp_final_energies):
    print(f"{p:<10.3f} {e_dep:>22.10f} {e_amp:>28.10f}")

## Interpretation

The physical effect of a noise model depends on the channel:

- **depolarizing noise** tends to randomize the state
- **amplitude damping** drives population loss toward lower computational-basis
  levels

Because these channels act differently, two scans with the same nominal noise
probability do not generally produce the same optimization behavior or the same
final energy.

## What this notebook showed

We:

- ran `VQE` for `H2` across a grid of noise probabilities
- compared depolarizing and amplitude-damping noise
- plotted final energies and absolute errors
- inspected representative convergence traces

This is the basic noisy-VQE scan workflow in the repository.

## Next steps

Good follow-ups are:

- scan both noise channels together
- repeat the study with a different ansatz
- compare noisy `VQE` and noisy `QPE`
- run multi-seed noise statistics rather than a single-seed scan